# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Data, features + the label: `declined_30d_future`

The baseline queue is one row per content item at a single **decision point `D`**. Every feature window ends at `D`; the label is a *future* outcome measured in the 30 days after `D`, so it was not knowable at decision time.

- **Table:** `fact_content_daily_performance` (warehouse). Iterating on the two month partitions that cover the window (`month=2026-05` = prior, `month=2026-06` = label). The `_sample` table is June-only, so it cannot feed the prior window on its own.
- **Decision point:** `D = 2026-05-31`. Facts run through `2026-06-30` (freshest 3 days cut), so the 30-day label window `(D, D+30]` is fully observed.
- **Features:** prior-window aggregates from `(D-30, D]` (impressions, clicks, CTR, weighted position, engagement) plus static content metadata from `dim_content` (type, intent, word count, age, staleness). All are knowable at `D`.
- **Label:** `declined_30d_future` = 1 when GSC impressions in `(D, D+30]` are under 80% of impressions in `(D-30, D]`, else 0. Pages without 30 days of history or with too few observed days are **not labelable** and get `NaN` — they are not counted as "not declined".
- The cache `work/outputs/baseline_features.csv` holds features + label on this exact slice, so rule iteration in section 1 is a single CSV load. This is the label and slice the w05 model will use.

In [3]:
import os
import getpass
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import numpy as np
import pandas as pd

load_dotenv()  # repo .env carries HF_TOKEN locally; Colab falls back to the prompt below.
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

# Iterate on the two partitions that cover the feature + label window (D-30 .. D+30).
# Final pass: FACT = TABLES["fact_daily"] once, then re-run and cache to work/outputs/.
FACT = (
    "read_parquet(['"
    + f"{REL}/fact_content_daily_performance/month=2026-05/*.parquet',"
    + f"'{REL}/fact_content_daily_performance/month=2026-06/*.parquet'])"
)
print("FACT =", FACT)

FACT = read_parquet(['hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet','hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-06/*.parquet'])


In [2]:
# Skill verification: COUNT(*) + MIN/MAX(report_date) on the table I iterate on.
row = con.sql(f"SELECT COUNT(*) AS n, MIN(report_date), MAX(report_date) FROM {FACT}").fetchone()
print(f"FACT: {row[0]:,} rows | {row[1]} -> {row[2]}  (May + Jun 2026 partitions)")

n_full = con.sql(f"SELECT COUNT(*) FROM {TABLES['fact_daily']}").fetchone()[0]
print(f"full fact table: {n_full:,} rows (skill expects 78,835,655)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FACT: 23,381,448 rows | 2026-05-01 -> 2026-06-30  (May + Jun 2026 partitions)


full fact table: 78,835,655 rows (skill expects 78,835,655)


In [3]:
# Grain probe: client x content x date should be one row. w03 found 6,390 dups in the full table.
dups = con.sql(f"""
    SELECT COUNT(*) FROM (
        SELECT client_hash_id, content_hash_id, report_date
        FROM {FACT}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print(f"duplicate rows at the grain: {dups:,}")

# Panel coverage: the unbalanced panel means per-client windows, never one global calendar window.
clients = con.sql(f"""
    SELECT c.client_hash_id, c.has_gsc_access, c.gsc_data_start,
           MAX(f.report_date) AS last_report
    FROM {TABLES['dim_clients']} c
    LEFT JOIN {TABLES['fact_daily']} f USING (client_hash_id)
    GROUP BY 1, 2, 3
""").df()
with_gsc = clients[clients["has_gsc_access"]]
print(f"clients: {len(clients)} | with GSC access: {len(with_gsc)}")
print("gsc_data_start range:", with_gsc["gsc_data_start"].min(), "->", with_gsc["gsc_data_start"].max())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate rows at the grain: 6,390


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

clients: 104 | with GSC access: 67
gsc_data_start range: 2025-01-27 00:00:00 -> 2026-06-02 00:00:00


In [4]:
END_DATE = pd.Timestamp(con.sql(f"SELECT MAX(report_date) FROM {FACT}").fetchone()[0])
D = END_DATE - pd.Timedelta(days=30)
print("facts end:", END_DATE.date())
print("decision point D:", D.date())
print("prior window (features):", (D - pd.Timedelta(days=30)).date(), "->", D.date())
print("label window (future):", (D + pd.Timedelta(days=1)).date(), "->", END_DATE.date())
print("label fully observed because D + 30 == END")

facts end: 2026-06-30
decision point D: 2026-05-31
prior window (features): 2026-05-01 -> 2026-05-31
label window (future): 2026-06-01 -> 2026-06-30
label fully observed because D + 30 == END


#### Build the label + prior-window features

One SQL pass over the daily facts: per-content prior-30d and future-30d windows, prior-window rule features (clicks, weighted position, engagement), and static content metadata — with the same client/content coverage filters as the label. Context columns repeat per content, so they are `ANY_VALUE()`-ed, never summed. The pandas step derives the label and the derived rate/age features.

In [5]:
SQL = f"""
WITH dedup AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY client_hash_id, content_hash_id, report_date) AS rn
    FROM {FACT}
),
ok AS (SELECT * FROM dedup WHERE rn = 1),
client_bounds AS (
    SELECT c.client_hash_id, c.gsc_data_start, MAX(f.report_date) AS last_report
    FROM {TABLES['dim_clients']} c
    LEFT JOIN {TABLES['fact_daily']} f USING (client_hash_id)
    GROUP BY 1, 2
),
windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        -- label windows
        SUM(CASE WHEN f.gsc_data_available
                 AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}'
                 THEN f.gsc_impressions ELSE 0 END) AS prior_imp,
        COUNT(CASE WHEN f.gsc_data_available
                   AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}'
                   THEN 1 END) AS prior_obs_days,
        SUM(CASE WHEN f.gsc_data_available
                 AND f.report_date > DATE '{D.date()}' AND f.report_date <= DATE '{D.date()}' + INTERVAL 30 DAY
                 THEN f.gsc_impressions ELSE 0 END) AS future_imp,
        COUNT(CASE WHEN f.gsc_data_available
                   AND f.report_date > DATE '{D.date()}' AND f.report_date <= DATE '{D.date()}' + INTERVAL 30 DAY
                   THEN 1 END) AS future_obs_days,
        -- prior-window rule features (GSC)
        SUM(CASE WHEN f.gsc_data_available
                 AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}'
                 THEN f.gsc_clicks ELSE 0 END) AS prior_clicks,
        COUNT(CASE WHEN f.gsc_data_available
                   AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}'
                   AND f.gsc_impressions > 0 THEN 1 END) AS prior_days_with_impressions,
        CASE WHEN SUM(CASE WHEN f.gsc_data_available
                           AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}'
                           THEN f.gsc_impressions ELSE 0 END) > 0
             THEN SUM(CASE WHEN f.gsc_data_available
                           AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}'
                           THEN f.gsc_sum_position ELSE 0 END)
                / SUM(CASE WHEN f.gsc_data_available
                           AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}'
                           THEN f.gsc_impressions ELSE 0 END)
             ELSE NULL END AS prior_position,
        -- first half of the prior window (D-30, D-15]: trend features for Rule 2
        SUM(CASE WHEN f.gsc_data_available
                 AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}' - INTERVAL 15 DAY
                 THEN f.gsc_impressions ELSE 0 END) AS prior_imp_h1,
        COUNT(CASE WHEN f.gsc_data_available
                   AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}' - INTERVAL 15 DAY
                   THEN 1 END) AS prior_obs_h1,
        CASE WHEN SUM(CASE WHEN f.gsc_data_available
                           AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}' - INTERVAL 15 DAY
                           THEN f.gsc_impressions ELSE 0 END) > 0
             THEN SUM(CASE WHEN f.gsc_data_available
                           AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}' - INTERVAL 15 DAY
                           THEN f.gsc_sum_position ELSE 0 END)
                / SUM(CASE WHEN f.gsc_data_available
                           AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}' - INTERVAL 15 DAY
                           THEN f.gsc_impressions ELSE 0 END)
             ELSE NULL END AS prior_pos_h1,
        -- second half of the prior window (D-15, D]: trend features for Rule 2
        SUM(CASE WHEN f.gsc_data_available
                 AND f.report_date > DATE '{D.date()}' - INTERVAL 15 DAY AND f.report_date <= DATE '{D.date()}'
                 THEN f.gsc_impressions ELSE 0 END) AS prior_imp_h2,
        COUNT(CASE WHEN f.gsc_data_available
                   AND f.report_date > DATE '{D.date()}' - INTERVAL 15 DAY AND f.report_date <= DATE '{D.date()}'
                   THEN 1 END) AS prior_obs_h2,
        CASE WHEN SUM(CASE WHEN f.gsc_data_available
                           AND f.report_date > DATE '{D.date()}' - INTERVAL 15 DAY AND f.report_date <= DATE '{D.date()}'
                           THEN f.gsc_impressions ELSE 0 END) > 0
             THEN SUM(CASE WHEN f.gsc_data_available
                           AND f.report_date > DATE '{D.date()}' - INTERVAL 15 DAY AND f.report_date <= DATE '{D.date()}'
                           THEN f.gsc_sum_position ELSE 0 END)
                / SUM(CASE WHEN f.gsc_data_available
                           AND f.report_date > DATE '{D.date()}' - INTERVAL 15 DAY AND f.report_date <= DATE '{D.date()}'
                           THEN f.gsc_impressions ELSE 0 END)
             ELSE NULL END AS prior_pos_h2,
        -- prior-window rule features (GA4; zero-filled when the client has no GA4 yet)
        SUM(CASE WHEN f.ga4_data_available
                 AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}'
                 THEN f.ga4_engaged_sessions ELSE 0 END) AS prior_engaged_sessions,
        SUM(CASE WHEN f.ga4_data_available
                 AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}'
                 THEN f.ga4_pageviews ELSE 0 END) AS prior_pageviews,
        COUNT(CASE WHEN f.ga4_data_available
                   AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY AND f.report_date <= DATE '{D.date()}'
                   THEN 1 END) AS prior_ga4_obs_days,
        -- static content metadata (known before D; ANY_VALUE, never SUM)
        ANY_VALUE(c.content_type) AS content_type,
        ANY_VALUE(c.main_intent) AS main_intent,
        ANY_VALUE(c.word_count) AS word_count,
        ANY_VALUE(c.search_volume) AS search_volume,
        ANY_VALUE(c.competition) AS competition,
        ANY_VALUE(c.cpc) AS cpc,
        ANY_VALUE(c.content_created_date) AS content_created_date,
        ANY_VALUE(c.content_updated_date) AS content_updated_date
    FROM ok f
    JOIN client_bounds b USING (client_hash_id)
    LEFT JOIN {TABLES['dim_content']} c USING (client_hash_id, content_hash_id)
    WHERE b.gsc_data_start <= DATE '{D.date()}' - INTERVAL 30 DAY
      AND b.last_report >= DATE '{D.date()}' + INTERVAL 30 DAY
      AND c.content_created_date <= DATE '{D.date()}' - INTERVAL 30 DAY
      AND f.report_date > DATE '{D.date()}' - INTERVAL 30 DAY
      AND f.report_date <= DATE '{D.date()}' + INTERVAL 30 DAY
    GROUP BY 1, 2
)
SELECT * FROM windowed
"""

label = con.sql(SQL).df()
print(f"content rows from the window: {len(label):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

content rows from the window: 333,275


In [6]:
MIN_PRIOR_IMP = 100   # min-volume floor: tiny pages are noise
MIN_OBS_DAYS = 7      # a real page shows up on several days, not one spike
DECLINE_FACTOR = 0.8  # future < 80% of prior counts as declined

label["labelable"] = (
    (label["prior_imp"] >= MIN_PRIOR_IMP)
    & (label["prior_obs_days"] >= MIN_OBS_DAYS)
    & (label["future_obs_days"] >= MIN_OBS_DAYS)
)
label["declined_30d_future"] = np.nan
label.loc[label["labelable"], "declined_30d_future"] = (
    label.loc[label["labelable"], "future_imp"]
    < DECLINE_FACTOR * label.loc[label["labelable"], "prior_imp"]
).astype(int)
label["declined_30d_future"] = label["declined_30d_future"].astype("Int64")

# Derived rule features — all computable at D.
label["prior_ctr"] = label["prior_clicks"] / label["prior_imp"].replace(0, np.nan)
label["content_age_days"] = (D - pd.to_datetime(label["content_created_date"])).dt.days
label["days_since_update"] = (D - pd.to_datetime(label["content_updated_date"])).dt.days.clip(lower=0)

n_labelable = int(label["labelable"].sum())
print(f"content rows: {len(label):,} | labelable: {n_labelable:,} "
      f"| filtered out: {len(label) - n_labelable:,}")
print(f"base rate of declined_30d_future: {label['declined_30d_future'].mean():.3f}")
label.head()

content rows: 333,275 | labelable: 100,785 | filtered out: 232,490
base rate of declined_30d_future: 0.655


,client_hash_id,content_hash_id,prior_imp,prior_obs_days,future_imp,future_obs_days,prior_clicks,prior_days_with_impressions,prior_position,prior_engaged_sessions,...,search_volume,competition,cpc,content_created_date,content_updated_date,labelable,declined_30d_future,prior_ctr,content_age_days,days_since_update
0,client_06d356715a8ff3b6,content_4b1a326e78f25818,769.0,30,213.0,27,1.0,30,14.224967,0.0,...,880,0.00,0.00,2026-04-06,2026-06-15,True,1,0.001300,55,0
1,client_08a6a72ff48e62c0,content_005925bc6dd04761,12.0,7,1.0,1,0.0,7,26.250000,0.0,...,90,0.31,4.43,2025-04-22,2026-04-21,False,<NA>,0.000000,404,40
2,client_08a6a72ff48e62c0,content_00cf93265c7b7509,3943.0,30,4344.0,30,51.0,30,3.918336,0.0,...,0,0.00,0.00,2025-09-19,2026-05-20,True,0,0.012934,254,11
3,client_08a6a72ff48e62c0,content_052464ae7eb418af,92.0,25,8.0,8,0.0,25,69.543478,0.0,...,0,0.00,0.00,2025-12-18,2026-04-19,False,<NA>,0.000000,164,42
4,client_08a6a72ff48e62c0,content_05d719178941817f,40.0,20,14.0,9,1.0,20,23.250000,0.0,...,10,0.10,0.00,2025-07-14,2026-05-20,False,<NA>,0.025000,321,11


In [7]:
# Verification: one row per content, label window fully observed, feature windows end at D.
aligned = label[label["labelable"]]
print("one row per content:", label["content_hash_id"].is_unique)
print("future window observed days — min/median/max:",
      aligned["future_obs_days"].min(), int(aligned["future_obs_days"].median()),
      aligned["future_obs_days"].max())
print("prior  window observed days — min/median/max:",
      aligned["prior_obs_days"].min(), int(aligned["prior_obs_days"].median()),
      aligned["prior_obs_days"].max())
print(f"labelable clients: {aligned['client_hash_id'].nunique()}")

# Cache features + label on the same slice (derived output, never a raw dataset).
cwd = Path.cwd()
OUT = cwd / "work" / "outputs" if (cwd / "work").is_dir() else cwd.parent / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
cache_cols = [
    "client_hash_id", "content_hash_id",
    "prior_imp", "prior_clicks", "prior_ctr", "prior_obs_days", "prior_days_with_impressions",
    "prior_position", "prior_engaged_sessions", "prior_pageviews", "prior_ga4_obs_days",
    "prior_imp_h1", "prior_imp_h2", "prior_obs_h1", "prior_obs_h2", "prior_pos_h1", "prior_pos_h2",
    "content_type", "main_intent", "word_count", "search_volume", "competition", "cpc",
    "content_age_days", "days_since_update",
    "future_imp", "future_obs_days", "labelable", "declined_30d_future",
]
label[cache_cols].to_csv(OUT / "baseline_features.csv", index=False)
print("cached to:", OUT / "baseline_features.csv")

one row per content: True
future window observed days — min/median/max: 7 30 30
prior  window observed days — min/median/max: 7 30 30
labelable clients: 41


cached to: /Users/wyatt/Documents/programming/flyrank/work/outputs/baseline_features.csv


**Window alignment — the only overlap trap.** The label lives in June 2026, the final month of the panel. `fact_content_query_90d` covers a fixed 90-day window that *includes* those months, so its `impressions_90d` / `*_last30` columns CONTAIN the label period — using them as features would be leakage (`docs/data-dictionary.md:141`). This baseline builds features only from the daily fact table, with windows ending at `D`, so it has no overlap. If query-mix features are added later, only the `*_prev30` columns are safe.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output. The feature cache is loaded below — iterate on the rule in this cell. Keep the baseline frozen once model work starts.*

In [8]:
# One load for all rule iteration: features known at D, plus the label on the same slice.
cwd = Path.cwd()
OUT = cwd / "work" / "outputs" if (cwd / "work").is_dir() else cwd.parent / "outputs"
rules = pd.read_csv(OUT / "baseline_features.csv")
rules = rules[rules["labelable"]].reset_index(drop=True)
print(f"{len(rules):,} labelable content rows to rank | declined base rate: {rules['declined_30d_future'].mean():.3f}")
rules.head()

100,785 labelable content rows to rank | declined base rate: 0.655


,client_hash_id,content_hash_id,prior_imp,prior_clicks,prior_ctr,prior_obs_days,prior_days_with_impressions,prior_position,prior_engaged_sessions,prior_pageviews,...,word_count,search_volume,competition,cpc,content_age_days,days_since_update,future_imp,future_obs_days,labelable,declined_30d_future
0,client_23a62021009f63c4,content_b3a79426c5e220a1,951.0,4.0,0.004206,30,30,23.115668,0.0,19.0,...,2586.0,0.0,0.0,0.0,207,0,575.0,30,True,1.0
1,client_23a62021009f63c4,content_b63094bd417908e9,15531.0,7.0,0.000451,30,30,35.801751,1.0,34.0,...,5370.0,0.0,0.0,0.0,144,95,14396.0,30,True,0.0
2,client_23a62021009f63c4,content_b8b51ec7b2768867,118.0,1.0,0.008475,27,27,13.669492,0.0,8.0,...,3383.0,0.0,0.0,0.0,116,95,309.0,27,True,0.0
3,client_23a62021009f63c4,content_bddfdd871aa09fbe,112236.0,4.0,0.000036,30,30,1.158701,1.0,62.0,...,3352.0,0.0,0.0,0.0,139,0,49074.0,30,True,1.0
4,client_23a62021009f63c4,content_c203b0cca77865dc,579.0,1.0,0.001727,30,30,37.685665,0.0,23.0,...,3052.0,0.0,0.0,0.0,102,11,306.0,30,True,1.0


### Rule 1: *stage x visible*

Intuition: the longer an article hasn't been updated, the more likely it needs to be refreshed.

In [9]:
# Rule 1, coded: score = old * visible * prior_imp  (no fitted weights)
MIN_STALE = 180         # "old": available at least a year at D
MIN_VISIBLE_IMP = 500  # "visible": still getting impressions in the prior window

stale  = (rules["days_since_update"] >= MIN_STALE).astype(int)
seen = (rules["prior_imp"] >= MIN_VISIBLE_IMP).astype(int)
rules["score"] = stale * seen * rules["prior_imp"]      # readable on purpose

def rule1_reasons(row):
    reasons = []
    if row["content_age_days"] >= MIN_STALE:
        reasons.append("old")
    if row["prior_imp"] >= MIN_VISIBLE_IMP:
        reasons.append("visible")
    return "|".join(reasons) if reasons else "not_old_not_visible"

rules["reason_codes"] = rules.apply(rule1_reasons, axis=1)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = rules["declined_30d_future"].astype(float).values
for k in (20, 50, 100):
    p_k = precision_at_k(rules["score"], y, k)
    print(f"Rule 1  Precision@{k}: {p_k:.3f}   (base rate: {y.mean():.3f})")

rules.sort_values("score", ascending=False).head(10)

Rule 1  Precision@20: 0.900   (base rate: 0.655)
Rule 1  Precision@50: 0.900   (base rate: 0.655)
Rule 1  Precision@100: 0.770   (base rate: 0.655)


,client_hash_id,content_hash_id,prior_imp,prior_clicks,prior_ctr,prior_obs_days,prior_days_with_impressions,prior_position,prior_engaged_sessions,prior_pageviews,...,competition,cpc,content_age_days,days_since_update,future_imp,future_obs_days,labelable,declined_30d_future,score,reason_codes
84810,client_20259bd6705d81d4,content_0d2aaf57d7146812,11551.0,59.0,0.005108,30,30,8.308545,1.0,71.0,...,0.00,0.0,222,184,8806.0,30,True,1.0,11551.0,old|visible
88943,client_20259bd6705d81d4,content_66d1fffc91f4f029,6988.0,16.0,0.002290,30,30,22.955209,0.0,20.0,...,0.00,0.0,222,185,2776.0,30,True,1.0,6988.0,old|visible
54246,client_20259bd6705d81d4,content_ac4e2d9d3bbb06de,6788.0,10.0,0.001473,30,30,20.073659,1.0,16.0,...,0.00,0.0,222,185,4115.0,30,True,1.0,6788.0,old|visible
65403,client_20259bd6705d81d4,content_097459d155cccb26,5864.0,19.0,0.003240,30,30,24.877217,1.0,20.0,...,0.00,0.0,222,185,3060.0,30,True,1.0,5864.0,old|visible
55738,client_20259bd6705d81d4,content_7907f31e6f1c1bfb,5842.0,44.0,0.007532,30,30,7.710202,0.0,56.0,...,0.00,0.0,222,185,5639.0,30,True,0.0,5842.0,old|visible
91982,client_20259bd6705d81d4,content_f2df5a8a9057783e,5693.0,15.0,0.002635,30,30,19.365712,0.0,18.0,...,0.00,0.0,222,185,2550.0,30,True,1.0,5693.0,old|visible
10639,client_157ffe4d4a595515,content_19daa2f24df1882d,5076.0,17.0,0.003349,30,30,16.300236,0.0,53.0,...,0.33,0.0,272,237,3429.0,25,True,1.0,5076.0,old|visible
34576,client_20259bd6705d81d4,content_283e87bc4e224d58,5064.0,26.0,0.005134,30,30,9.748815,0.0,46.0,...,0.00,0.0,222,185,3267.0,30,True,1.0,5064.0,old|visible
62111,client_20259bd6705d81d4,content_3f962469cb61a7c9,4610.0,28.0,0.006074,30,30,10.994577,2.0,46.0,...,0.00,0.0,222,185,2521.0,30,True,1.0,4610.0,old|visible
64659,client_20259bd6705d81d4,content_47da45b084a73115,4228.0,2.0,0.000473,30,30,50.281930,0.0,3.0,...,0.00,0.0,222,185,2508.0,30,True,1.0,4228.0,old|visible


### Conclusion: CONFIRMED
Rule 1 beat baseline for Precision@20, Precision@50, and Precision@100.

### Rule 2: *falling impressions x slipping position*

Plain words: a page is worth refreshing when its impressions are falling across the prior month **and** its
position is slipping too — the classic fading star. The prior window `(D-30, D]` is split in half: **falling** =
the second half has under 80% of the first half's impressions; **slipping** = the weighted position got worse
(a higher number). Reason codes: `impressions_falling`, `position_slipping`.

In [10]:
# Rule 2, coded: falling impressions AND slipping position, ranked by impressions lost.
# The prior month (D-30, D] is split in half — falling = the second half loses more than
# 20% of the first half's impressions; slipping = the weighted position gets worse (higher).

MIN_HALF_OBS = 5      # each half must be observed on several days or the ratio is noise
FALL_FACTOR = 0.8     # second half < 80% of first half counts as falling

h1_ok = rules["prior_obs_h1"] >= MIN_HALF_OBS
h2_ok = rules["prior_obs_h2"] >= MIN_HALF_OBS
pos_ok = (rules["prior_pos_h1"].notna() & rules["prior_pos_h2"].notna()
          & (rules["prior_pos_h1"] > 0) & (rules["prior_pos_h2"] > 0))

rules["imp_ratio"] = rules["prior_imp_h2"] / rules["prior_imp_h1"].replace(0, np.nan)
rules["pos_delta"] = rules["prior_pos_h2"] - rules["prior_pos_h1"]

falling  = (h1_ok & h2_ok & (rules["imp_ratio"] < FALL_FACTOR)).astype(int)
slipping = (pos_ok & (rules["pos_delta"] > 0)).astype(int)
rules["score"] = falling * slipping * (rules["prior_imp_h1"] - rules["prior_imp_h2"])

def rule2_reasons(row):
    reasons = []
    if row["prior_obs_h1"] >= MIN_HALF_OBS and row["prior_obs_h2"] >= MIN_HALF_OBS and row["imp_ratio"] < FALL_FACTOR:
        reasons.append("impressions_falling")
    if pd.notna(row["pos_delta"]) and row["pos_delta"] > 0:
        reasons.append("position_slipping")
    return "|".join(reasons) if reasons else "not_falling_or_slipping"

rules["reason_codes"] = rules.apply(rule2_reasons, axis=1)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = rules["declined_30d_future"].astype(float).values
for k in (20, 50, 100):
    p_k = precision_at_k(rules["score"], y, k)
    print(f"Rule 2  Precision@{k}: {p_k:.3f}   (base rate: {y.mean():.3f})")

rules.sort_values("score", ascending=False).head(10)

Rule 2  Precision@20: 0.900   (base rate: 0.655)
Rule 2  Precision@50: 0.920   (base rate: 0.655)
Rule 2  Precision@100: 0.920   (base rate: 0.655)


,client_hash_id,content_hash_id,prior_imp,prior_clicks,prior_ctr,prior_obs_days,prior_days_with_impressions,prior_position,prior_engaged_sessions,prior_pageviews,...,content_age_days,days_since_update,future_imp,future_obs_days,labelable,declined_30d_future,score,reason_codes,imp_ratio,pos_delta
4607,client_0fa64a184f18a4a0,content_11bf4c33adea7bdc,87854.0,1.0,0.000011,30,30,8.257973,0.0,1.0,...,81,11,107215.0,30,True,0.0,85162.0,impressions_falling|position_slipping,0.015559,5.541640
70318,client_1a730cb2640a1abf,content_39e19a3ec2d95f9d,178306.0,2.0,0.000011,30,30,10.094859,0.0,12.0,...,130,6,6777.0,30,True,1.0,74922.0,impressions_falling|position_slipping,0.408264,0.104956
24365,client_23a62021009f63c4,content_40baa8f1016f5742,79496.0,293.0,0.003686,30,30,3.577123,6.0,1681.0,...,290,0,22969.0,30,True,1.0,50634.0,impressions_falling|position_slipping,0.221794,4.565833
13057,client_8ddc46da5414ffd8,content_5913241ddeecf3f1,147526.0,721.0,0.004887,30,30,2.368986,0.0,0.0,...,52,0,15058.0,30,True,1.0,46402.0,impressions_falling|position_slipping,0.521451,0.261860
72226,client_73cda7b4e4f265ea,content_8c6d8360702fff11,49466.0,8.0,0.000162,30,30,6.227934,0.0,20.0,...,473,0,6378.0,30,True,1.0,45710.0,impressions_falling|position_slipping,0.039464,1.626135
14811,client_23a62021009f63c4,content_86c96002dd5c69aa,79649.0,148.0,0.001858,30,30,21.200768,14.0,446.0,...,207,0,19370.0,30,True,1.0,45335.0,impressions_falling|position_slipping,0.274547,25.203928
99195,client_e5c2aa26a8598242,content_72524cabb2854075,61673.0,62.0,0.001005,30,30,6.018663,1.0,107.0,...,117,0,29467.0,30,True,1.0,43081.0,impressions_falling|position_slipping,0.177482,4.475030
4002,client_73cda7b4e4f265ea,content_fc7f6650dba17854,48221.0,3.0,0.000062,30,30,9.967566,2.0,25.0,...,454,0,1708.0,30,True,1.0,43073.0,impressions_falling|position_slipping,0.056389,1.622951
27736,client_73cda7b4e4f265ea,content_b5f3280f8d894862,53336.0,88.0,0.001650,30,30,4.072446,3.0,158.0,...,321,0,8032.0,30,True,1.0,40278.0,impressions_falling|position_slipping,0.139488,2.406554
3505,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,116296.0,187.0,0.001608,30,30,5.424924,4.0,293.0,...,471,0,63860.0,30,True,1.0,36908.0,impressions_falling|position_slipping,0.518185,1.506330


### Conclusion: CONFIRMED
Rule 2 beat baseline for PRecision@20, Precision@50, and Precision@100.

### Rule 1 vs Rule 2
Rule 2 outperformed Rule 1 (specifically for Precision@100).

## 2. Build the ranked queue (writes the CSV)

The frozen baseline is **Rule 2** — falling impressions x position (P@100 0.92 beats Rule 1's 0.77 and spans more clients). Score every labelable row, rank descending, write `work/outputs/baseline_action_score.csv`.

In [13]:
# Section 2: frozen baseline = Rule 2 (falling impressions x position). Fresh load from cache,
# so this cell is self-contained and rerunnable regardless of which rule cell ran last.

rules = pd.read_csv(OUT / "baseline_features.csv")
rules = rules[rules["labelable"]].reset_index(drop=True)

MIN_HALF_OBS = 5      # each half must be observed on several days or the ratio is noise
FALL_FACTOR = 0.8     # second half < 80% of first half counts as falling

h1_ok = rules["prior_obs_h1"] >= MIN_HALF_OBS
h2_ok = rules["prior_obs_h2"] >= MIN_HALF_OBS
pos_ok = (rules["prior_pos_h1"].notna() & rules["prior_pos_h2"].notna()
          & (rules["prior_pos_h1"] > 0) & (rules["prior_pos_h2"] > 0))

rules["imp_ratio"] = rules["prior_imp_h2"] / rules["prior_imp_h1"].replace(0, np.nan)
rules["pos_delta"] = rules["prior_pos_h2"] - rules["prior_pos_h1"]

falling  = (h1_ok & h2_ok & (rules["imp_ratio"] < FALL_FACTOR)).astype(int)
slipping = (pos_ok & (rules["pos_delta"] > 0)).astype(int)
rules["score"] = falling * slipping * (rules["prior_imp_h1"] - rules["prior_imp_h2"])

def rule2_reasons(row):
    reasons = []
    if row["prior_obs_h1"] >= MIN_HALF_OBS and row["prior_obs_h2"] >= MIN_HALF_OBS and row["imp_ratio"] < FALL_FACTOR:
        reasons.append("IMPRESSIONS_FALLING")
    if pd.notna(row["pos_delta"]) and row["pos_delta"] > 0:
        reasons.append("POSITION_SLIPPING")
    return "|".join(reasons) if reasons else "not_falling_or_slipping"

rules["reason_codes"] = rules.apply(rule2_reasons, axis=1)

queue = rules.sort_values("score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

y = queue["declined_30d_future"].astype(float).values
order = np.argsort(-np.asarray(queue["score"]))
for k in (20, 50, 100):
    p_k = np.asarray(y)[order[:k]].mean()
    print(f"Baseline (Rule 2)  Precision@{k}: {p_k:.3f}   (base rate: {y.mean():.3f})")

cols = ["rank", "client_hash_id", "content_hash_id", "score", "reason_codes", "declined_30d_future"]
queue[cols].to_csv(OUT / "baseline_action_score.csv", index=False)
print("wrote", OUT / "baseline_action_score.csv", f"({len(queue):,} ranked rows)")

Baseline (Rule 2)  Precision@20: 0.900   (base rate: 0.655)
Baseline (Rule 2)  Precision@50: 0.920   (base rate: 0.655)
Baseline (Rule 2)  Precision@100: 0.920   (base rate: 0.655)
wrote /Users/wyatt/Documents/programming/flyrank/work/outputs/baseline_action_score.csv (100,785 ranked rows)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [14]:
# Top-20 review table for the frozen Rule 2 baseline.
# The label compares future_imp against the FULL prior window (prior_imp = h1 + h2), so
# fut/prior is the decline ratio that matters — h2/h1 shows the mid-month fall itself.
review = pd.read_csv(OUT / "baseline_action_score.csv").head(20)
feats = pd.read_csv(OUT / "baseline_features.csv")
feats = feats[feats["labelable"]]
review = review.merge(
    feats[["client_hash_id", "content_hash_id", "prior_imp", "prior_imp_h1", "prior_imp_h2",
           "future_imp", "prior_pos_h1", "prior_pos_h2", "content_age_days", "days_since_update"]],
    on=["client_hash_id", "content_hash_id"],
)
review["h2/h1"] = (review["prior_imp_h2"] / review["prior_imp_h1"]).round(3)
review["fut/prior"] = (review["future_imp"] / review["prior_imp"]).round(3)
review[["rank", "client_hash_id", "content_hash_id", "score", "reason_codes", "declined_30d_future",
        "prior_imp_h1", "prior_imp_h2", "future_imp", "prior_pos_h1", "prior_pos_h2",
        "content_age_days", "days_since_update", "h2/h1", "fut/prior"]].set_index("rank")

,client_hash_id,content_hash_id,score,reason_codes,declined_30d_future,prior_imp_h1,prior_imp_h2,future_imp,prior_pos_h1,prior_pos_h2,content_age_days,days_since_update,h2/h1,fut/prior
rank,,,,,,,,,,,,,,
1,client_0fa64a184f18a4a0,content_11bf4c33adea7bdc,85162.0,IMPRESSIONS_FALLING|POSITION_SLIPPING,0.0,86508.0,1346.0,107215.0,8.173071,13.714710,81,11,0.016,1.220
2,client_1a730cb2640a1abf,content_39e19a3ec2d95f9d,74922.0,IMPRESSIONS_FALLING|POSITION_SLIPPING,1.0,126614.0,51692.0,6777.0,10.064432,10.169388,130,6,0.408,0.038
3,client_23a62021009f63c4,content_40baa8f1016f5742,50634.0,IMPRESSIONS_FALLING|POSITION_SLIPPING,1.0,65065.0,14431.0,22969.0,2.748282,7.314115,290,0,0.222,0.289
4,client_8ddc46da5414ffd8,content_5913241ddeecf3f1,46402.0,IMPRESSIONS_FALLING|POSITION_SLIPPING,1.0,96964.0,50562.0,15058.0,2.279238,2.541098,52,0,0.521,0.102
5,client_73cda7b4e4f265ea,content_8c6d8360702fff11,45710.0,IMPRESSIONS_FALLING|POSITION_SLIPPING,1.0,47588.0,1878.0,6378.0,6.166197,7.792332,473,0,0.039,0.129
6,client_23a62021009f63c4,content_86c96002dd5c69aa,45335.0,IMPRESSIONS_FALLING|POSITION_SLIPPING,1.0,62492.0,17157.0,19370.0,15.771651,40.975578,207,0,0.275,0.243
7,client_e5c2aa26a8598242,content_72524cabb2854075,43081.0,IMPRESSIONS_FALLING|POSITION_SLIPPING,1.0,52377.0,9296.0,29467.0,5.344140,9.819170,117,0,0.177,0.478
8,client_73cda7b4e4f265ea,content_fc7f6650dba17854,43073.0,IMPRESSIONS_FALLING|POSITION_SLIPPING,1.0,45647.0,2574.0,1708.0,9.880934,11.503885,454,0,0.056,0.035
9,client_73cda7b4e4f265ea,content_b5f3280f8d894862,40278.0,IMPRESSIONS_FALLING|POSITION_SLIPPING,1.0,46807.0,6529.0,8032.0,3.777854,6.184408,321,0,0.139,0.151


### Top-20 hand review

All 20 picks passed both gates (impressions_falling + position_slipping) and span 9 clients. 18 of 20 actually declined in June — the two that did not (ranks 1 and 12) are the weak picks this review exists to catch. Action for every row: **refresh / republish the content**.

| # | Content | Reason code | Confidence | What would make it wrong |
|---|---------|-------------|------------|--------------------------|
| 1 | `content_11bf4c33` | IMPRESSIONS_FALLING,POSITION_SLIPPING | LOW | declined_30d_future = 0.0; not a real decline. |
| 2 | `content_39e19a3e` | IMPRESSIONS_FALLING,POSITION_SLIPPING | HIGH | 127k→52k→6.8k across both halves and June. |
| 3 | `content_40baa8f1` | IMPRESSIONS_FALLING,POSITION_SLIPPING | HIGH | 65k→14k→23k, position 2.7→7.3. A competitor sweep of its queries rather than a freshness problem — refresh may not win them back. |
| 4 | `content_5913241d` | IMPRESSIONS_FALLING,POSITION_SLIPPING | HIGH | 97k→51k→15k, position slip mild (2.3→2.5). A niche-wide algorithm change rather than something page-level. |
| 5 | `content_8c6d8360` | IMPRESSIONS_FALLING,POSITION_SLIPPING | HIGH | 48k→1.9k→6.4k, near-total collapse. The page was deindexed or redirected — refresh cannot fix a removed page. |
| 6 | `content_86c96002` | IMPRESSIONS_FALLING,POSITION_SLIPPING | HIGH | 63k→17k→19k, position 15.8→41. Demand for the target queries went away entirely. |
| 7 | `content_72524cab` | IMPRESSIONS_FALLING,POSITION_SLIPPING | MEDIUM-HIGH | 52k→9.3k→29k; June partly recovered (fut/prior 0.48). The recovery carries into July. |
| 8 | `content_fc7f6650` | IMPRESSIONS_FALLING,POSITION_SLIPPING | HIGH | 46k→2.6k→1.7k, sustained. A sitewide drop rather than a page-level problem. |
| 9 | `content_b5f3280f` | IMPRESSIONS_FALLING,POSITION_SLIPPING | HIGH | 47k→6.5k→8k. The whole client site lost rankings. |
| 10 | `content_fd2117c2` | IMPRESSIONS_FALLING,POSITION_SLIPPING | MEDIUM-HIGH | 77k→40k→64k; fut/prior 0.55 — declined, but June bounced back to 64k. The recovery continues. |
| 11 | `content_fa84f597` | IMPRESSIONS_FALLING,POSITION_SLIPPING | HIGH | 49k→12k→10k, position 13.7→27.2. Seasonality — June is the niche's low season, not a permanent loss. |
| 12 | `content_7918f4ac` | IMPRESSIONS_FALLING,POSITION_SLIPPING | **LOW** | **Weak pick** — June recovered to 55k (fut/prior 0.81); the mid-May dip was transient. |
| 13 | `content_e7b5dd4d` | IMPRESSIONS_FALLING,POSITION_SLIPPING | HIGH | 127k→92k→48k, sustained. A competitor published a stronger page. |
| 14 | `content_6f50bf27` | IMPRESSIONS_FALLING,POSITION_SLIPPING | MEDIUM-HIGH | 48k→14k→33k; fut/prior 0.54 — declined, but June rebounded to 33k. Recovery continues. |
| 15 | `content_f60372c9` | IMPRESSIONS_FALLING,POSITION_SLIPPING | MEDIUM-HIGH | 61k→28k→38k; fut/prior 0.42. The late-May dip was a blip that fully recovers. |
| 16 | `content_13bffb19` | IMPRESSIONS_FALLING,POSITION_SLIPPING | MEDIUM | 40k→6.9k→30k; fut/prior 0.65 — declined, close to the line, and June is recovering fast. The June uptrend continues. |
| 17 | `content_e7712e5a` | IMPRESSIONS_FALLING,POSITION_SLIPPING | HIGH | 62k→29k→13k, sustained. A sitewide algorithm or penalty issue. |
| 18 | `content_a14cf3f4` | IMPRESSIONS_FALLING,POSITION_SLIPPING | HIGH | 34k→1.5k→11k, near-total collapse. The page was removed or noindexed. |
| 19 | `content_e578ac84` | IMPRESSIONS_FALLING,POSITION_SLIPPING | MEDIUM | 82k→50k→100k; fut/prior 0.76 — declined only just under the 0.8 line, and June is back to 100k. The rebound is real; the flag is borderline. |
| 20 | `content_f7c9fcc2` | IMPRESSIONS_FALLING,POSITION_SLIPPING | MEDIUM-HIGH | 70k→37k→52k; fut/prior 0.49 — declined, but June partly recovered. Recovery continues into July. |

Pattern the weak picks expose: a crash inside the prior month that self-heals. The rule cannot yet distinguish a transient dip from a sustained decline — worth noting in the paper.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**#1**: declined_30d_future 0.0. Weird sharp decline but recovered following months.

**#12**: declined_30d_future 0.0. Another transient dip that instantly recovered.

In [15]:
# Leakage check — attack the frozen Rule 2 baseline before anyone else does.
# A feature is leaky if it could only be known after decision point D.

# 1) The rule's inputs are all prior-window aggregates ending at D — none are label-derived
#    or future-window columns.
LEAKY_FEATURE_COLS = ["future_imp", "future_obs_days", "declined_30d_future",
                      "trend_direction", "trend_pct"]
RULE2_INPUTS = ["prior_imp_h1", "prior_imp_h2", "prior_obs_h1", "prior_obs_h2",
                "prior_pos_h1", "prior_pos_h2"]
overlap = [c for c in RULE2_INPUTS if c in LEAKY_FEATURE_COLS]
print("1. rule inputs that are label-derived/future columns:", overlap if overlap else "none ✓")

# 2) No product flags or existing-system decision columns anywhere in the cache.
#    (future_imp / declined_30d_future ARE in the cache by design — they are the label,
#    used only for evaluation, never fed into the score.)
cache_cols = pd.read_csv(OUT / "baseline_features.csv", nrows=1).columns
bad = [c for c in cache_cols if "product" in c.lower() or "flag" in c.lower()]
print("2. product-flag / decision columns in cache:", bad if bad else "none ✓")

# 3) Window alignment: the two halves partition the prior window exactly, so both feature
#    windows end at D and nothing overlaps the label window (D, D+30].
leak = pd.read_csv(OUT / "baseline_features.csv")
leak = leak[leak["labelable"]]
gap = (leak["prior_imp"] - (leak["prior_imp_h1"] + leak["prior_imp_h2"])).abs().max()
print(f"3. max |prior_imp - (h1 + h2)|: {gap:g}",
      "→ halves partition the prior window; nothing bleeds into June ✓" if gap == 0 else "→ MISALIGNED ✗")

# 4) Positive control: rank by a deliberately LEAKY score (future impressions — which literally
#    contain the label) and precision must jump toward 1.0. If it stayed ~0.92, the harness
#    itself would be broken.
def p_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = leak["declined_30d_future"].astype(float).values
for k in (50, 100):
    leaky_p = p_at_k(-leak["future_imp"], y, k)   # smallest future first = most likely declined
    print(f"4. LEAK CONTROL (rank by -future_imp)  Precision@{k}: {leaky_p:.3f}  | honest Rule 2 @100: 0.920")

1. rule inputs that are label-derived/future columns: none ✓
2. product-flag / decision columns in cache: none ✓
3. max |prior_imp - (h1 + h2)|: 0 → halves partition the prior window; nothing bleeds into June ✓
4. LEAK CONTROL (rank by -future_imp)  Precision@50: 1.000  | honest Rule 2 @100: 0.920
4. LEAK CONTROL (rank by -future_imp)  Precision@100: 1.000  | honest Rule 2 @100: 0.920


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.